<a href="https://colab.research.google.com/github/kushalshah0/colab_tools/blob/main/NEPSE_Index_Scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import pandas as pd
import requests

# Target endpoint base URL
url = "https://www.sharesansar.com/index-history-data"

# Unified header context mimicking a conventional browser profile
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://www.sharesansar.com/index-history-data",
}

# Base tracking parameters (Mapped from your browser URLs)
base_params = {
    "index_id": "12",  # 12 is the internal structural key for the NEPSE Index
    "from": "1990-11-28",
    "to": "2026-07-27",
    "length": "20",  # Row window step size
}

all_records = []
start_offset = 0
draw_counter = 1
is_active = True

print("Initializing NEPSE History Extraction Cycle...")

while is_active:
    # Update pagination window tracking state dynamically on every iteration
    current_params = base_params.copy()
    current_params["start"] = str(start_offset)
    current_params["draw"] = str(draw_counter)
    # Cache-busting UNIX timestamp variant
    current_params["_"] = str(int(time.time() * 1000))

    try:
        # Utilize a GET sequence to bypass server endpoint filters
        response = requests.get(url, params=current_params, headers=headers)

        if response.status_code == 200:
            payload = response.json()
            data_rows = payload.get("data", [])

            # Drop out of the infinite loop if the row buffer yields empty blocks
            if not data_rows:
                print(
                    "Finished extraction: Reached final row block boundary."
                )
                is_active = False
                break

            all_records.extend(data_rows)
            print(
                f"Pulled Rows {start_offset} to {start_offset + len(data_rows)} | Total Gathered: {len(all_records)}"
            )

            # Shift pagination pointer forward by window row size
            start_offset += 20
            draw_counter += 1

            # Politeness delay buffer to prevent hitting firewalls (1.5-second sleep)
            time.sleep(1.5)

        elif response.status_code == 429:
            print("Server rate limits tripped. Sleeping loop for 10 seconds...")
            time.sleep(10)
        else:
            print(
                f"Request failed at offset {start_offset} with code: {response.status_code}"
            )
            is_active = False

    except Exception as e:
        print(f"An anomaly broken the runtime state: {e}")
        is_active = False

# Structure data elements safely post-extraction
if all_records:
    df_raw = pd.DataFrame(all_records)

    # Format the Dataframe array neatly to match terminal datasets
    final_dataset = pd.DataFrame(
        {
            "Date": df_raw["published_date"],
            "Open": df_raw["open"],
            "High": df_raw["high"],
            "Low": df_raw["low"],
            "Close": df_raw["current"],
            "Change": df_raw["change_"],
            "Pct_Change": df_raw["per_change"],
            "Turnover": df_raw["turnover"],
        }
    )

    # Sort sequentially from earliest to latest trading calendar days
    final_dataset = final_dataset.sort_values(by="Date", ascending=True)

    output_csv = "nepse_all_history.csv"
    final_dataset.to_csv(output_csv, index=False)
    print(f"\n✅ Processing complete! File saved as: '{output_csv}'")
    print(final_dataset.head(10))
else:
    print("No historical elements were compiled.")
